# TorchTitan-NPU 的 VarLen+CP 长序列容量实验

第 7 章已经介绍 FSDP、TP 与 CP 的集合通信。本章聚焦一个可验证的问题：当单条 S=16,384 的上下文使 no-CP 训练发生 OOM 时，Ulysses CP 如何通过 AllToAll 交换 sequence/head 布局，让多个 rank 共同承担 sequence-shaped activation。

VarLen 需要正确的全局隔离区间；CP 的两个 rank 必须共享这些边界。CP 的收益首先是容量扩展，而非默认的吞吐加速：理论、双 rank memory trace、OOM 证据和 AllToAll 暴露时间必须形成一条完整证据链。

---


## 教程进度回顾

| 章节 | 内容 | 状态 |
|---|---|---|
| 第 1 章 | SFT 概念 + Wordle 任务 | ✅ 已完成 |
| 第 2 章 | TorchTitan 框架 + 环境配置 | ✅ 已完成 |
| 第 3 章 | 数据准备 + 基线训练 + 推理评测 | ✅ 已完成 |
| 第 4 章 | 融合算子优化 + Profiling | ✅ 已完成 |
| 第 5 章 | Attention 公式、算子与 TorchTitan dispatch | ✅ 已完成 |
| 第 6 章 | Sequence Packing、VarLen Attention 与端到端效率 | ✅ 已完成 |
| 第 7 章 | FSDP、TP、CP 集合通信与通信量分析 | ✅ 已完成 |
| **第 8 章** | **VarLen+FSDP、VarLen+CP 与端到端 Profiling** | ← 当前 |

---


## 本章目标

完成本章后，你将能够：

- 解释 VarLen 隔离区间与 Ulysses CP sequence/head 重排之间的约束；
- 沿 TorchTitan-NPU 的实现路径定位 VarLen+CP backend、pre/post hook 与 AllToAll；
- 设计同一 `S=16,384`、batch、数据和 dtype 的 no-CP/CP2 容量实验；
- 用理论 activation、双 rank peak、OOM 和非重叠通信判读 CP 的收益与代价。

---


## 前置条件

- 完成第 6 章，理解 sequence packing、`cu_seq` 与 NPU TND VarLen Attention；
- 完成第 7 章，理解 FSDP 参数通信和 CP AllToAll 的数据布局变化；
- 能够运行 Qwen3-1.7B Wordle SFT recipe，并读取 TorchTitan 与 TorchTitan-NPU 源码；
- 当前环境可使用两张 Ascend NPU 运行训练和 profiler。

---


## 本章结构

| Notebook | 内容 |
|---|---|
| [08.01](08.01_chapter_intro.ipynb) | 章节介绍（本节） |
| [08.02](08.02_varlen_cp_interaction.ipynb) | VarLen+CP 长序列设计、全局边界与 NPU 实现 |
| [08.03](08.03_cp_long_sequence_capacity_profiling.ipynb) | S=16,384 的 CP 容量实验：理论、采集、双 rank trace 分析与结论 |
| [08.04](08.04_tp2_fsdp2_comparison.ipynb) | TP2/FSDP2 同 workload 对比 |
| [08.05](08.05_chapter_practice.ipynb) | 章节练习 |

---


## 容量实验的两条路线

| 路线 | parallelism | `seq_len` | attention backend | 关键通信 |
|---|---|---:|---|---|
| no-CP | `dp_replicate=1, dp_shard=2, cp=1` | 16,384 | NPU TND VarlenAttention | parameter all-gather / gradient reduce-scatter |
| CP2 | `dp_replicate=1, dp_shard=1, cp=2` | 16,384 | NPU TND VarlenAttention | 参数通信 + Q/K/V/output AllToAll |

当前 TorchTitan 上游的 Qwen3 `update_from_config()` 会拒绝 VarLen+CP；TorchTitan-NPU 通过 `NPUVarlenAttention`、`NPUVarlenUlyssesCP` 和 `sft_qwen3_1_7b_wordle_tnd` 支持这条路线。08.02 会定位这些仓库内实现。

---


## 固定实验卡与公平性

- Qwen3-1.7B、同一 Wordle parquet、bf16、两张 Ascend NPU；
- `seq_len=16384`、`local_batch_size=2`、global batch、数据顺序、activation checkpoint 与训练步数保持一致；
- no-CP 使用 `dp_shard=2, cp=1`，CP2 使用 `dp_shard=1, cp=2`；
- profile 同时采集两个 rank 的 Step 5（`profile_ranks=-1, profile_step_start=5, profile_step_end=6`）。

容量结论读取双 rank `memory_record.csv` 与 OOM；AllToAll 账本和关键路径读取 `communication.json` 与 `step_trace_time.csv`。TPS 不属于本实验，必须另做 profiler-off 的同 workload ablation。


In [ ]:
EXPERIMENT_CARD = {
    'model': 'Qwen3-1.7B',
    'dataset': 'assets/data/wordle',
    'dtype': 'bf16',
    'world_size': 2,
    'seq_len': {'no_cp': 16384, 'cp2': 16384},
    'local_batch_size': 2,
    'profile_step': 5,
    'notebooks': ('08.02 VarLen+CP', '08.03 CP capacity profiling', '08.04 TP/FSDP2', '08.05 practice'),
}
for key, value in EXPERIMENT_CARD.items():
    print(f'{key:>18}: {value}')


---

## 本章产物与验收

1. `sft_qwen3_1_7b_wordle_tnd` 的 28 层均使用 `NPUVarlenAttention`，且 mask 为 `block_causal`；
2. CP2 trace 验证全局区间边界、TND kernel 与 attention 前后 AllToAll；
3. no-CP/CP2 使用同一 S=16,384 workload，分别保留 OOM/训练日志和双 rank profiler 输出；
4. 结论报告理论 sequence-shaped activation、实际 peak/是否 OOM 和 AllToAll 暴露，而不把历史 raw sample/s 写成 CP 加速常数。

本章要学生带走的是：**CP 用通信换长上下文容量。**若 no-CP 已 OOM、CP2 可运行，结论是“该 workload 从不可运行变为可运行”；不要把 OOM 前显存伪造成完整 peak reduction 百分比。


## 练习

1. （判断题）CP ranks 共同处理同一批上下文，计算 global batch size 时不能把 CP degree 当作数据并行副本数。

2. （判断题）若 QKV 的理论本地占用在 CP2 下减半，则完整 NPU peak 必然恰好减半。

3. （单选题）本章的主要容量对比对象是什么？
    A. 同一 S=16,384 workload 的 no-CP 与 CP2
    B. VarLen+FSDP 4096 与 VarLen+CP 8192 的 TPS 排名
    C. Dense 推理与量化推理
    D. TP 与 Expert Parallel

4. （多选题）形成 CP 容量结论需要哪些证据？
    A. 理论 sequence-shaped activation
    B. 双 rank peak 和 OOM 记录
    C. AllToAll 账本与 step 级非重叠通信
    D. 历史 raw sample/s


In [ ]:
!cat ./answer/08.01_answer.txt
